# 05 · Simulation & Scenario Projection

Monte Carlo simulation of terminal portfolio values across a user-selected horizon. Default uses a multivariate normal fit on daily log-returns; optionally a historical bootstrap preserves fat tails.

In [ ]:
# Bootstrap path so the notebook finds the src/ package
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
print("Project root:", ROOT)


In [ ]:
from data_loader import load_all, LoaderConfig
from feature_engineering import compute_fund_features
from recommendation import RecommendationEngine, UserInput

data = load_all(LoaderConfig(offline_mode=True, lookback_days=1260))
features = compute_fund_features(data["universe"], data["navs"], data["benchmarks"])
engine = RecommendationEngine(data["universe"], features, data["navs"])
user = UserInput(investment_amount=1_000_000, horizon_years=7, risk_appetite="Medium")
portfolios = engine.recommend(user)
list(portfolios.keys())

### Terminal value distribution per portfolio

In [ ]:
import numpy as np, matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i, (name, p) in enumerate(portfolios.items()):
    ax = axes[i]
    term = p.projection["terminal_values"]
    ax.hist(term / 1e5, bins=60, color="#4B7BEC", alpha=0.75)
    ax.axvline(np.percentile(term, 5)/1e5, color="red", ls="--", lw=1, label="p5")
    ax.axvline(np.percentile(term, 50)/1e5, color="black", ls="-", lw=1, label="median")
    ax.axvline(np.percentile(term, 95)/1e5, color="green", ls="--", lw=1, label="p95")
    ax.set_title(name); ax.set_xlabel("Terminal value (₹ Lakh)")
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[-1].axis("off")
plt.tight_layout(); plt.show()

### Percentile paths — Balanced portfolio

In [ ]:
p = portfolios["Balanced"]
paths = p.projection["percentile_paths"]
fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(paths.index, paths["p5"]/1e5, paths["p95"]/1e5, alpha=0.2, color="#4B7BEC", label="5–95 %")
ax.fill_between(paths.index, paths["p25"]/1e5, paths["p75"]/1e5, alpha=0.35, color="#4B7BEC", label="25–75 %")
ax.plot(paths.index, paths["p50"]/1e5, color="black", lw=1.5, label="median")
ax.set_xlabel("Trading day"); ax.set_ylabel("Portfolio value (₹ Lakh)")
ax.set_title("Balanced portfolio — MC percentile paths")
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

### Summary table

In [ ]:
import pandas as pd
rows = []
for name, p in portfolios.items():
    rows.append({
        "Portfolio": name,
        "E[R]": p.expected_return,
        "Vol": p.volatility,
        "Sharpe": p.sharpe_ratio,
        "MDD": p.max_drawdown,
        "VaR95": p.var_95,
        "CAGR p5":  p.projection["cagr"]["worst_case_p5"],
        "CAGR p50": p.projection["cagr"]["median_p50"],
        "CAGR p95": p.projection["cagr"]["best_case_p95"],
    })
pd.DataFrame(rows).set_index("Portfolio").style.format({c:"{:.2%}" for c in
    ["E[R]","Vol","MDD","VaR95","CAGR p5","CAGR p50","CAGR p95"]}).format({"Sharpe":"{:.2f}"})